**Introduction**

This Collab contains a walkthrough Implementation of a BERT based pretrained model for Sentiment Analysis. The Dataset consists of Customer Product Reviews fetched from `Flipkart`.

The use of the BERT pre-trained model for this purpose stems from its ability to understand the context of words in text by considering the entire phrase in which they appear. This deep understanding makes BERT highly effective for tasks like sentiment analysis, where the sentiment of a review may hinge on subtle nuances in language

In [ ]:
import pandas as pd
import numpy as np

df_reviews = pd.read_csv("/content/drive/MyDrive/Data Mining/flipkart_product_reviews.csv")

df_reviews.rename(columns={'Review': 'Summary', 'Summary': 'Customer Review'}, inplace=True)


In [ ]:
df_reviews.head()

,product_name,product_price,Rate,Summary,Customer Review,Sentiment
0,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,5,super!,great cooler excellent air flow and for this p...,positive
1,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,5,awesome,best budget 2 fit cooler nice cooling,positive
2,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,3,fair,the quality is good but the power of air is de...,positive
3,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,1,useless product,very bad product its a only a fan,negative
4,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,3,fair,ok ok product,neutral


### Preprocessing

In [ ]:
df_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205052 entries, 0 to 205051
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   product_name     205052 non-null  object
 1   product_price    205052 non-null  object
 2   Rate             205052 non-null  object
 3   Summary          180388 non-null  object
 4   Customer Review  205041 non-null  object
 5   Sentiment        205052 non-null  object
dtypes: object(6)
memory usage: 9.4+ MB


In [ ]:
df_reviews.isnull().sum()


product_name           0
product_price          0
Rate                   0
Summary            24664
Customer Review       11
Sentiment              0
dtype: int64

In [ ]:
df_reviews.dropna(inplace=True, axis=0)


In [ ]:
df_reviews['Sentiment'].unique()

array(['positive', 'negative', 'neutral'], dtype=object)

In [ ]:
#
def clean_data(df, column_name):
    df[column_name] = df[column_name].str.replace(r'\?\?\?\?\?\?', '')
    return df

df_reviews = clean_data(df_reviews, 'product_name')

<ipython-input-7-d2bf920555ba>:3: FutureWarning: The default value of regex will change from True to False in a future version.
  df[column_name] = df[column_name].str.replace(r'\?\?\?\?\?\?', '')


In [ ]:
import re

def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text, re.I|re.A)
    text = text.lower()
    return text

df_reviews['Customer Review'] = df_reviews['Customer Review'].apply(clean_text)

In [ ]:
df_reviews.head()

,product_name,product_price,Rate,Summary,Customer Review,Sentiment
0,"Candes 12 L Room/Personal Air Cooler(White, Bl...",3999,5,super!,great cooler excellent air flow and for this p...,positive
1,"Candes 12 L Room/Personal Air Cooler(White, Bl...",3999,5,awesome,best budget fit cooler nice cooling,positive
2,"Candes 12 L Room/Personal Air Cooler(White, Bl...",3999,3,fair,the quality is good but the power of air is de...,positive
3,"Candes 12 L Room/Personal Air Cooler(White, Bl...",3999,1,useless product,very bad product its a only a fan,negative
4,"Candes 12 L Room/Personal Air Cooler(White, Bl...",3999,3,fair,ok ok product,neutral


**Encode Sentiments**

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df_reviews['Sentiment_encoded'] = label_encoder.fit_transform(df_reviews['Sentiment'])

print("Mapping between original sentiment and encoded labels :\n")
for label, encoded_label in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"{label}: {encoded_label}")

Mapping between original sentiment and encoded labels :

negative: 0
neutral: 1
positive: 2


In [ ]:
df_reviews.head()

,product_name,product_price,Rate,Summary,Customer Review,Sentiment,Sentiment_encoded
0,"Candes 12 L Room/Personal Air Cooler(White, Bl...",3999,5,super!,great cooler excellent air flow and for this p...,positive,2
1,"Candes 12 L Room/Personal Air Cooler(White, Bl...",3999,5,awesome,best budget fit cooler nice cooling,positive,2
2,"Candes 12 L Room/Personal Air Cooler(White, Bl...",3999,3,fair,the quality is good but the power of air is de...,positive,2
3,"Candes 12 L Room/Personal Air Cooler(White, Bl...",3999,1,useless product,very bad product its a only a fan,negative,0
4,"Candes 12 L Room/Personal Air Cooler(White, Bl...",3999,3,fair,ok ok product,neutral,1


In [ ]:
df_reviews.shape

(180379, 7)

In [ ]:
# Training data
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df_reviews, test_size=0.3, random_state=42)


In [ ]:

Reviews = train_df['Customer Review']
Target = train_df['Sentiment_encoded']

# Test data
test_reviews = test_df['Customer Review']
test_targets = test_df['Sentiment_encoded']

In [ ]:
x_val, x_test, y_val, y_test = train_test_split(test_reviews,
													test_targets,
													test_size=0.5,
													stratify = test_targets)


**BERT Training**

In [ ]:
import tensorflow as tf
from transformers import BertTokenizer, TFBertForSequenceClassification

#Tokenize and encode the data using the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
max_len= 128
# Tokenize and encode the sentences
X_train_encoded = tokenizer.batch_encode_plus(Reviews.tolist(),
											padding=True,
											truncation=True,
											max_length = max_len,
											return_tensors='tf')

X_val_encoded = tokenizer.batch_encode_plus(x_val.tolist(),
											padding=True,
											truncation=True,
											max_length = max_len,
											return_tensors='tf')

X_test_encoded = tokenizer.batch_encode_plus(x_test.tolist(),
											padding=True,
											truncation=True,
											max_length = max_len,
											return_tensors='tf')


In [ ]:
# Intialize the model
model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
model.summary()

Model: "tf_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  109482240 
                                                                 
 dropout_37 (Dropout)        multiple                  0 (unused)
                                                                 
 classifier (Dense)          multiple                  2307      
                                                                 
Total params: 109484547 (417.65 MB)
Trainable params: 109484547 (417.65 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


In [ ]:
import os
import tensorflow as tf


history = model.fit(
	[X_train_encoded['input_ids'], X_train_encoded['token_type_ids'], X_train_encoded['attention_mask']],
	Target,
	validation_data=(
	[X_val_encoded['input_ids'], X_val_encoded['token_type_ids'], X_val_encoded['attention_mask']],y_val),
	batch_size=32,
	epochs=3
)


Epoch 1/3


Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
3946/3946 [==============================] - 1272s 295ms/step - loss: 2.9657 - accuracy: 0.8159 - val_loss: 2.9756 - val_accuracy: 0.8154
Epoch 2/3
3946/3946 [==============================] - 1126s 285ms/step - loss: 2.9640 - accuracy: 0.8161 - val_loss: 2.9756 - val_accuracy: 0.8154
Epoch 3/3
3946/3946 [==============================] - 1120s 284ms/step - loss: 2.9640 - accuracy: 0.8161 - val_loss: 2.9756 - val_accuracy: 0.8154


In [ ]:
#Evaluate the model on the test data
test_loss, test_accuracy = model.evaluate(
	[X_test_encoded['input_ids'], X_test_encoded['token_type_ids'], X_test_encoded['attention_mask']],
	y_test
)
print(f'loss: {test_loss}, accuracy: {test_accuracy}')


846/846 [==============================] - 96s 100ms/step - loss: 2.9750 - accuracy: 0.8154
loss: 2.9749691486358643, accuracy: 0.8154267072677612


In [ ]:
path = '/content/drive/MyDrive/Data Mining'
# Save tokenizer
tokenizer.save_pretrained(path +'/Tokenizer')

# Save model
model.save_pretrained(path +'/Model')


In [ ]:
# Load tokenizer
bert_tokenizer = BertTokenizer.from_pretrained(path +'/Tokenizer')

# Load model
bert_model = TFBertForSequenceClassification.from_pretrained(path +'/Model')


Some layers from the model checkpoint at /content/drive/MyDrive/Data Mining/Model were not used when initializing TFBertForSequenceClassification: ['dropout_37']
- This IS expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at /content/drive/MyDrive/Data Mining/Model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassification for predictions without further training.


In [ ]:
pred = bert_model.predict(
	[X_test_encoded['input_ids'], X_test_encoded['token_type_ids'], X_test_encoded['attention_mask']])

# pred is of type TFSequenceClassifierOutput
logits = pred.logits

pred_labels = tf.argmax(logits, axis=1)

pred_labels = pred_labels.numpy()

label = {

	0: 'negative',
  1: 'neutral',
  2: 'positive',
}

# Maps the predicted labels to their corresponding strings using the label dictionary
pred_labels = [label[i] for i in pred_labels]
Actual = [label[i] for i in y_test]

print('Predicted Label :', pred_labels[:10])
print('Actual Label :', Actual[:10])


846/846 [==============================] - 106s 111ms/step
Predicted Label : ['positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive']
Actual Label : ['positive', 'positive', 'positive', 'negative', 'positive', 'negative', 'positive', 'positive', 'positive', 'positive']


In [ ]:
def predict_sentiment(Review, Tokenizer=bert_tokenizer, Model=bert_model):
	# Converts Review to a list if it's not already a list
	if not isinstance(Review, list):
		Review = [Review]

	Input_ids, Token_type_ids, Attention_mask = Tokenizer.batch_encode_plus(Review,
																			padding=True,
																			truncation=True,
																			max_length=128,
																			return_tensors='tf').values()
	prediction = Model.predict([Input_ids, Token_type_ids, Attention_mask])

	pred_labels = tf.argmax(prediction.logits, axis=1)

	pred_labels = [label[i] for i in pred_labels.numpy().tolist()]
	return pred_labels


In [ ]:
Review ='''Product is great and suprisingly concrete. Number 1 customer.
          '''
predict_sentiment(Review)


1/1 [==============================] - 0s 110ms/step


['positive']